# Paper tables and cited numbers from committed results (A25)

Renders the manuscript's remaining tables and every scalar cited in its running text from the committed
`results/*.json`, `results/*.csv` and the Sybil list, so that rerunning `25` and then `99_tie_out_revision`
refreshes and re-checks them. It fits no model; it calls `sp.build_master_df` once, for the bounty-list statistics
and the large gas provision trees.

Tables (`tables/*.tex`, booktabs `tabular` fragments): `rev_data_labels`, `rev_taxonomy`, `rev_levels`,
`rev_per_category`, `rev_family_ablation`, `rev_provision_ablation_by_category`, `rev_entity_level`,
`rev_temporal_holdout`, `rev_union_split_appendix`, `rev_search_top`. Named scalars: `tables/rev_text_numbers.json`.

**Results**: `results/25_paper_tables.json`.

## Setup

In [ ]:
import hashlib, json, os, platform, re, shutil, subprocess, tempfile, warnings
import numpy as np
import pandas as pd
import sybil_pipeline as sp
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
assert not sp.CODE_COMMIT.endswith('-dirty'), f'Run from a clean working tree (CODE_COMMIT={sp.CODE_COMMIT})'
print('Code commit:', sp.CODE_COMMIT)
PLATFORM = dict(machine=platform.machine(), processor=platform.processor(), system=platform.system(),
                python=platform.python_version(), cpu_count=os.cpu_count(), n_jobs=sp.N_JOBS)
SOURCES = ['00_data_pipeline', '02_hyperparameter_search', '07_sensitivity_label_vintage', '08_ablation_families',
           '11_split_comparison', '13_random_forest_sybil', '14_entity_level_recall', '15_split_tree_report',
           '16_label_robustness_initial_list', '17_models_10_splits', '19_ablation_by_category', '20_temporal_holdout',
           '22_report_dependence', '03_xgboost_sybil', '04_lightgbm_sybil']
pv = sp.provenance(SOURCES)
print(pv.to_string(index=False))
assert pv[pv.notebook != '14_entity_level_recall']['dependencies_unchanged'].all(), pv
R = {n: sp.load_results(n) for n in SOURCES}
FLAG14 = R['14_entity_level_recall']['inputs']['predictions_match_committed_results']
if not (pv.set_index('notebook').loc['14_entity_level_recall', 'dependencies_unchanged'] and FLAG14):
    print('WARNING: results/14 ran on predictions that do not reproduce the committed 03/04/06 '
          f'(predictions_match_committed_results = {FLAG14}); its table is marked UNVERIFIED until it is rerun.')
TAB_DIR = 'tables'; os.makedirs(TAB_DIR, exist_ok=True)
TABLES = {}

## Helpers (copied from `23`)

In [ ]:
def platform_of(n):
    p = R[n].get('platform') or R[n].get('inputs', {}).get('platform')
    return f"{p['machine']} / {p['system']}" if p else 'not recorded (machine of the committed 00-12 results)'
def source(names):
    return [dict(results=f'results/{n}.json', code_commit=R[n]['code_commit'], platform=platform_of(n)) for n in names]
def note_platforms(names):
    plats = {platform_of(n) for n in names}
    return None if len(plats) == 1 else 'mixes platforms: ' + '; '.join(f'{n}: {platform_of(n)}' for n in names)
fmt3 = lambda x: f'{x:.3f}'
pm = lambda m, s: f'{m:.3f} $\\pm$ {s:.3f}'
cnt = lambda x: f'{int(round(x)):,}'
pct = lambda x, d=1: f'{100 * x:.{d}f}\\%'
sgn = lambda x: '0.000' if round(x, 3) == 0 else f'{x:+.3f}'
def pval(p):
    return '$<$0.001' if p < 0.001 else f'{p:.3f}'
def write_table(name, colspec, header, rows, comment, names, extra_files=()):
    """booktabs tabular fragment; rows are lists of cell strings, or a str for a full-width line ('\\midrule' or a section)."""
    files = [f'results/{n}.json' for n in names] + list(extra_files)
    lines = [f'% {comment}', f'% Generated by 25_paper_tables.ipynb at {sp.CODE_COMMIT} from ' + ', '.join(files),
             f'\\begin{{tabular}}{{{colspec}}}', '\\toprule'] + ([' & '.join(header) + ' \\\\', '\\midrule'] if header else [])
    for r in rows:
        lines.append(r if isinstance(r, str) else ' & '.join(r) + ' \\\\')
    lines += ['\\bottomrule', '\\end{tabular}', '']
    text = '\n'.join(lines)
    with open(os.path.join(TAB_DIR, f'{name}.tex'), 'w') as f:
        f.write(text)
    TABLES[name] = dict(file=f'{TAB_DIR}/{name}.tex', sources=source(names) + [dict(file=x) for x in extra_files],
                        platform_note=note_platforms(names))
    print(f'--- {name}.tex'); print(text)
    return text
def row_of(text, first):
    hits = [l for l in text.splitlines() if l.startswith(first + ' &')]
    assert len(hits) == 1, (first, hits); return hits[0]
def cells_of(line):
    return [x.strip() for x in line.rstrip(' \\').split(' & ')]
r17 = R['17_models_10_splits']; t17 = pd.DataFrame(r17['tests'])
def tst(model, against, m):
    return t17[(t17.metric == m) & (t17.model == model) & (t17.against == against)].iloc[0]

## Data: the master table and the Sybil list

In [ ]:
df, _, _ = sp.build_master_df('./data', verbose=False)
SYB_PATH = sp.data_paths('./data')['sybil']
fcfs = pd.read_csv(SYB_PATH); fcfs.columns = fcfs.columns.str.lower().str.strip()
fcfs['address'] = fcfs['address'].str.lower()
syb = df[df['sybil'] == 1].merge(fcfs, left_on='addr', right_on='address', how='left', validate='one_to_one')
assert syb['commonwealth report link'].notna().all() and len(syb) == R['00_data_pipeline']['n_sybil']
rep_counts = syb['commonwealth report link'].value_counts(); who_counts = syb['reward address1'].value_counts()
ts = pd.to_datetime(fcfs['timestamp'], utc=True)
BOUNTY = dict(list_rows=len(fcfs), reports=int(rep_counts.size), reporters=int(who_counts.size),
              top10_reports_share=float(rep_counts.head(10).sum() / len(syb)), top10_reporters_share=float(who_counts.head(10).sum() / len(syb)),
              first_flagged=str(ts.min().date()), last_flagged=str(ts.max().date()))
EXPECTED = dict(reports=311, reporters=145, top10_reports_share=0.36, top10_reporters_share=0.45)
print(pd.DataFrame([dict(statistic=k, computed=BOUNTY[k], expected=v) for k, v in EXPECTED.items()]).to_string(index=False))
assert BOUNTY['reports'] == 311 and BOUNTY['reporters'] == 145
assert abs(BOUNTY['top10_reports_share'] - 0.36) < 0.005 and abs(BOUNTY['top10_reporters_share'] - 0.45) < 0.005, BOUNTY
# Large gas provision trees (over 100 wallets; the 34 that the tree split places the same way for every seed)
tsz = df.groupby('provision_tree')['sybil'].agg(['size', 'sum'])
big_t = tsz[tsz['size'] > 100]
LARGE_TREES = dict(n_trees=int(len(big_t)), wallets=int(big_t['size'].sum()), sybils=int(big_t['sum'].sum()),
                   wallet_share=float(big_t['size'].sum() / len(df)), sybil_share=float(big_t['sum'].sum() / df['sybil'].sum()))
print('Trees over 100 wallets:', LARGE_TREES)

## 1. The two label sets (`rev_data_labels`)

In [ ]:
r00, r16 = R['00_data_pipeline'], R['16_label_robustness_initial_list']
cat00 = r00['categories']; c16 = r16['counts']; bc16 = {x['category']: x for x in r16['by_category']}
N, NS = r00['n_addresses'], r00['n_sybil']
rows = [['Source and period', f"bounty reports accepted, flagged {BOUNTY['first_flagged']} to {BOUNTY['last_flagged']}",
         'LayerZero, Nansen, Chaos Labs; published 18 May 2024'],
        ['Addresses in list (all chains)', cnt(BOUNTY['list_rows']), cnt(c16['list_size'])],
        ['Ethereum interactors in list', f"{cnt(NS)} ({pct(NS / N, 2)})", f"{cnt(c16['interactors_on_list'])} ({pct(c16['interactors_on_list'] / N, 2)})"]]
for c in ['IxL', 'IxE', 'IxI', 'No provider']:
    rows.append([f'\\quad {c}', cnt(cat00[c]['sybil']), cnt(bc16[c]['on_list'])])
rows += [['Overlap with the other list', cnt(c16['overlap_with_fcfs_list']), cnt(c16['overlap_with_fcfs_list'])],
         ['Bounty reports / reporters', f"{cnt(BOUNTY['reports'])} / {cnt(BOUNTY['reporters'])}", '--'],
         ['Label in the standard setup', 'Sybil (positive)', 'non-Sybil (negative)']]
assert sum(cat00[c]['sybil'] for c in cat00) == NS and sum(bc16[c]['on_list'] for c in bc16) == c16['interactors_on_list']
write_table('rev_data_labels', 'lrr', ['', 'Bounty list', 'Initial list'], rows,
            'The two LayerZero label sets among the 434,786 Ethereum interactors. Initial list: review_support/initial_list/.',
            ['00_data_pipeline', '16_label_robustness_initial_list'], extra_files=[SYB_PATH])

## 2. Taxonomy (`rev_taxonomy`)

In [ ]:
rows = []
for c in ['IxL', 'IxE', 'IxI', 'No provider']:
    x = cat00[c]
    rows.append([c, cnt(x['n']), pct(x['n'] / N), cnt(x['sybil']), pct(x['sybil'] / x['n'], 2)])
tot_n, tot_s = sum(x['n'] for x in cat00.values()), sum(x['sybil'] for x in cat00.values())
assert tot_n == N == 434_786 and tot_s == NS == 18_211
rows += ['\\midrule', ['Total', cnt(tot_n), pct(1.0), cnt(tot_s), pct(tot_s / tot_n, 2)]]
write_table('rev_taxonomy', 'lrrrr', ['Category', 'Addresses', 'Share', 'Sybils', 'Sybil rate'], rows,
            'Taxonomy of interactors by first funder: labeled (IxL), another interactor (IxI), an unlabeled non-interactor (IxE).',
            ['00_data_pipeline'])

## 3. Dependence levels (`rev_levels`)

In [ ]:
r15, r22 = R['15_split_tree_report'], R['22_report_dependence']
ps15 = pd.DataFrame(r15['report_overlap']['per_split'])
L1 = ps15[(ps15.split == 'random') & (ps15.seed == 42)].iloc[0]; L2 = ps15[(ps15.split == 'tree') & (ps15.seed == 42)].iloc[0]
ii = r15['interpretation_inputs']; cs22 = r22['composition_summary']
assert int(L1['report']) == ii['report_overlap_random_seed42'] and int(L2['report']) == ii['report_overlap_tree_seed42']
assert int(L1['test_sybils']) == int(L2['test_sybils']) == ii['test_sybils']
rows = [['1. Address split', 'address', cnt(L1['test_sybils']), cnt(L1['tree']), cnt(L1['report'])],
        ['2. Tree split', 'gas provision tree', cnt(L2['test_sybils']), cnt(L2['tree']),
         f"{cnt(L2['report'])} (10 splits: {ii['report_overlap_tree_10seeds_mean']:,.1f} $\\pm$ {ii['report_overlap_tree_10seeds_sd']:.1f})"],
        ['3. Report-mates held out', 'tree and bounty report', f"{cs22['k_a_sybils_min']:,}--{cs22['k_a_sybils_max']:,} per fold", '0', '0']]
write_table('rev_levels', 'llrrr', ['Level', 'Unit kept whole', 'Test Sybils', 'Sharing a tree', 'Sharing a report'], rows,
            f"Test Sybils sharing a gas provision tree or a bounty report with a training Sybil. Levels 1-2: seed-42 split (results/15); level 3: held-out condition of the paired design (results/22; zero by construction, asserted there). In its seen condition {cs22['mate_coverage_mean']:.1%} of test Sybils have a report-mate in training.",
            ['15_split_tree_report', '22_report_dependence'])

## 4. Per-category performance over the ten tree splits (`rev_per_category`)

In [ ]:
bc = pd.DataFrame(r17['by_category']); si = pd.DataFrame(r17['split_info']).set_index('split_seed')
g = bc.groupby(['split_seed', 'model'])[['n', 'sybil']].sum()
for (s, mdl), r in g.iterrows():
    assert r['n'] == si.loc[s, 'n_test'], (s, mdl)
assert (g.groupby('split_seed')['sybil'].nunique() == 1).all()          # same test Sybils for every model of a split
PERCAT = {}
rows = []
for model in ['LightGBM', 'XGBoost']:
    for c in ['IxL', 'IxI', 'IxE', 'No provider']:
        t = bc[(bc.model == model) & (bc.category == c)]
        assert len(t) == 10
        PERCAT[(model, c)] = {m: (float(t[m].mean()), float(t[m].std(ddof=1))) for m in ['precision', 'recall', 'f1', 'auroc']}
        PERCAT[(model, c)]['sybils'] = float(t['sybil'].mean())
        rows.append([model if c == 'IxL' else '', c, f"{t['sybil'].mean():,.1f}"] + [pm(*PERCAT[(model, c)][m]) for m in ['precision', 'recall', 'f1', 'auroc']])
    if model == 'LightGBM':
        rows.append('\\midrule')
write_table('rev_per_category', 'llrrrrr', ['Model', 'Category', 'Test Sybils', 'Precision', 'Recall', 'F1', 'AUROC'], rows,
            'Mean $\\pm$ sample SD over 10 stratified group splits (one model seed each), at each split\'s validation threshold. No provider has about 7 test Sybils per split; its estimates are unreliable.',
            ['17_models_10_splits'])
L = lambda c, m: PERCAT[('LightGBM', c)][m]
print('Sanity (LightGBM, 10-split means): '
      f"IxL F1 {L('IxL', 'f1')[0]:.3f} ± {L('IxL', 'f1')[1]:.3f}, recall {L('IxL', 'recall')[0]:.3f}; "
      f"IxI F1 {L('IxI', 'f1')[0]:.3f} ± {L('IxI', 'f1')[1]:.3f}, recall {L('IxI', 'recall')[0]:.3f}, precision {L('IxI', 'precision')[0]:.3f}; "
      f"IxE F1 {L('IxE', 'f1')[0]:.3f} ± {L('IxE', 'f1')[1]:.3f}, recall {L('IxE', 'recall')[0]:.3f}, precision {L('IxE', 'precision')[0]:.3f}")
X = lambda c, m: PERCAT[('XGBoost', c)][m]
print('XGBoost: ' + '; '.join(f"{c} F1 {X(c, 'f1')[0]:.3f}, recall {X(c, 'recall')[0]:.3f}" for c in ['IxL', 'IxI', 'IxE']))
PERCAT_NOTE = ('The seed-42 per-category values in an earlier draft (IxI F1 0.534, IxE F1 0.265) are more favorable than these '
               '10-split means.')

## 5. Feature-family ablation (`rev_family_ablation`)

In [ ]:
r08 = R['08_ablation_families']; p08 = pd.DataFrame(r08['per_split']); s08 = r08['summary']
CONF = ['All features'] + [c for c in s08 if c.startswith('Without ') and c != 'Without all provision-network families'] + \
       [c for c in s08 if c == 'Without all provision-network families']
rows, FAMROWS = [], {}
for c in CONF:
    t = p08[p08.config == c]
    assert len(t) == 10
    d = t['test_f1_change']
    cells = [cnt(t['n_features'].iloc[0]), pm(t['test_f1'].mean(), t['test_f1'].std(ddof=1)),
             '--' if c == 'All features' else f"{sgn(d.mean())} $\\pm$ {d.std(ddof=1):.3f}",
             pm(t['test_ap'].mean(), t['test_ap'].std(ddof=1)), pm(t['test_auroc'].mean(), t['test_auroc'].std(ddof=1)),
             '--' if c == 'All features' else f"{int((d < 0).sum())}/10"]
    assert abs(t['test_f1'].mean() - s08[c]['test_f1_mean']) < 1e-12
    FAMROWS[c] = cells; rows.append([c] + cells)
write_table('rev_family_ablation', 'lrrrrrr', ['Configuration', 'Features', 'Test F1', '$\\Delta$F1', 'Test AP', 'Test AUROC', 'F1 lower'], rows,
            'LightGBM (02 parameters), one model seed, 10 stratified group splits; mean $\\pm$ SD; $\\Delta$F1 against all features, paired by split.',
            ['08_ablation_families'])

## 6. Provision-network ablation by category (`rev_provision_ablation_by_category`)

In [ ]:
d19 = pd.DataFrame(R['19_ablation_by_category']['diffs']); d19 = d19[d19.metric == 'f1']
CATS = ['All', 'IxL', 'IxI', 'IxE']
rows, PROV = [], {}
for w in d19['without'].unique():
    t = d19[d19.without == w].set_index('category')
    PROV[w] = {c: t.loc[c] for c in CATS}
    rows.append([w] + [f"{sgn(t.loc[c, 'diff_mean'])} $\\pm$ {t.loc[c, 'diff_sd']:.3f}" for c in CATS])
    rows.append(['\\quad full model higher; $p$'] + [f"{int(t.loc[c, 'all_higher_n'])}/10; {pval(t.loc[c, 'p_corrected'])}" for c in CATS])
t6 = write_table('rev_provision_ablation_by_category', 'lrrrr', ['Removed', 'All', 'IxL', 'IxI', 'IxE'], rows,
                 'Test F1, all features minus the configuration without the named features (LightGBM, one model seed, 10 splits), mean $\\pm$ SD; then splits where the full model is higher and the corrected resampled t-test p-value as stored by 19.',
                 ['19_ablation_by_category'])

## 7. Entity-level outcomes (`rev_entity_level`)

In [ ]:
r14 = R['14_entity_level_recall']; t14 = pd.DataFrame(r14['table'])
t14 = t14[(t14.model == 'LightGBM') & (t14.threshold_label == 'validation')]
plat14 = platform_of('14_entity_level_recall')
rows = []
for e in ['E1 gas provision tree', 'E2 bounty report', 'E2 bounty report (all Sybils in test)', 'E3 reporter']:
    r = t14[(t14.entity == e) & (t14.bucket == 'all')].iloc[0]
    rows.append([e, cnt(r['entities']), cnt(r['sybils']), fmt3(r['any_hit']), fmt3(r['majority_hit']), fmt3(r['full_hit']),
                 fmt3(r['within_recall_weighted']), fmt3(r['residual_allocation_share']), fmt3(r['entities_majority_allocation_unflagged_share'])])
rows += ['\\midrule', '\\multicolumn{9}{l}{\\emph{E1 gas provision tree, by test Sybils per tree}} \\\\']
for b in ['1', '2–5', '6–20', '21–100', '>100']:
    r = t14[(t14.entity == 'E1 gas provision tree') & (t14.bucket == b)]
    if len(r):
        r = r.iloc[0]
        rows.append([f"\\quad {b.replace('–', '--').replace('>', '$>$')}", cnt(r['entities']), cnt(r['sybils']), '--', '--', '--',
                     fmt3(r['within_recall_weighted']), '--', '--'])
flag_txt = f"predictions_match_committed_results = {FLAG14}; platform {plat14}"
write_table('rev_entity_level', 'lrrrrrrrr',
            ['Entity', 'Entities', 'Sybils', 'Any hit', 'Majority', 'Full', 'Recall', 'Unflagged ZRO', '$\\geq$50\\% unflagged'], rows,
            ('UNVERIFIED: ' if not FLAG14 else '') + 'LightGBM at the validation threshold, test partition (results/14). ' + flag_txt,
            ['14_entity_level_recall'])

## 8. Temporal holdout (`rev_temporal_holdout`)

In [ ]:
r20 = R['20_temporal_holdout']; res20 = pd.DataFrame(r20['results']).set_index('late_share_target'); co20 = pd.DataFrame(r20['cohorts']).set_index('late_share_target')
M20 = ['f1', 'recall', 'ap', 'auroc']
rows = []
for q in sorted(res20.index):
    r, c = res20.loc[q], co20.loc[q]
    rows.append([pct(q, 0), str(c['cutoff_utc'])[:10], pct(c['early_sybil_rate'], 2), pct(c['late_sybil_rate'], 2)] +
                [fmt3(r[f'temporal_{m}']) for m in M20] + [fmt3(r[f'reference_{m}']) for m in M20])
FIELDS20 = [f'temporal_{m}' for m in M20] + [f'reference_{m}' for m in M20] + ['cohorts.cutoff_utc', 'cohorts.early_sybil_rate', 'cohorts.late_sybil_rate']
write_table('rev_temporal_holdout', 'lllrrrrrrrrr',
            ['Late', 'From', 'Sybil early', 'Sybil late', 'F1', 'Recall', 'AP', 'AUROC', 'F1$^{r}$', 'Recall$^{r}$', 'AP$^{r}$', 'AUROC$^{r}$'], rows,
            f"Trained on the early cohort (70/30 group split), tested on the late cohort; r: the seed-42 tree-split model on its test rows in the same late cohort ({r20['reference']['split']}).",
            ['20_temporal_holdout'])

## 9. The superseded union split (`rev_union_split_appendix`)

In [ ]:
bp = pd.DataFrame(r15['composition']['by_partition'])
rows = []
for _, r in bp.iterrows():
    rows.append([r['split'], r['partition'], cnt(r['rows']), cnt(r['sybils']), cnt(r['sybils_in_BIG']), cnt(r['reports'])])
rows += ['\\midrule', ['\\emph{Union split, test}', '\\emph{Model}', '\\emph{F1}', '\\emph{AP}', '\\emph{AUROC}', '']]
for mdl in R['15_split_tree_report']['models']:
    rows.append(['seed 42 (3 seeds)', mdl['name'].split(' (')[0], fmt3(mdl['f1']), fmt3(mdl['ap']), fmt3(mdl['auroc']), '--'])
s15 = r15['summary']
rows.append(['10 splits (1 seed)', 'LightGBM', pm(s15['union_test_f1_mean'], s15['union_test_f1_std']),
             pm(s15['union_test_ap_mean'], s15['union_test_ap_std']), pm(s15['union_test_auroc_mean'], s15['union_test_auroc_std']), '--'])
rows.append(['10 splits, tree split', 'LightGBM', pm(s15['tree_test_f1_mean'], s15['tree_test_f1_std']),
             pm(s15['tree_test_ap_mean'], s15['tree_test_ap_std']), pm(s15['tree_test_auroc_mean'], s15['tree_test_auroc_std']), '--'])
write_table('rev_union_split_appendix', 'llrrrr', ['Split', 'Partition', 'Rows', 'Sybils', 'In large comps', 'Reports'], rows,
            'Superseded by the paired design (notebook 22): training diversity collapses. Union split = gas provision tree joined with bounty report; large comps = components over 5% of Sybils.',
            ['15_split_tree_report'])

## 10. Hyperparameter searches (`rev_search_top`)

In [ ]:
SEARCH = {'XGBoost': ('results/02_search_xgboost.csv', ['learning_rate', 'max_depth', 'subsample', 'colsample_bytree', 'min_child_weight'],
                      R['02_hyperparameter_search']['xgb_selected'], dict(learning_rate=0.3, max_depth=6, subsample=1.0, colsample_bytree=1.0, min_child_weight=1)),
          'LightGBM': ('results/02_search_lightgbm.csv', ['num_leaves', 'learning_rate', 'subsample', 'reg_lambda'],
                       R['02_hyperparameter_search']['lgbm_selected'], dict(num_leaves=31, learning_rate=0.1, subsample=1.0, reg_lambda=0.0)),
          'Random Forest': ('results/13_search_random_forest.csv', ['max_features', 'min_samples_leaf', 'max_depth'],
                            {k: R['13_random_forest_sybil']['params'][k] for k in ['max_features', 'min_samples_leaf', 'max_depth']}, None)}
ABBR = dict(learning_rate='lr', max_depth='depth', subsample='sub', colsample_bytree='col', min_child_weight='mcw', num_leaves='leaves',
            reg_lambda='$\\lambda$', max_features='mf', min_samples_leaf='leaf')
INT_HP = {'max_depth', 'min_child_weight', 'num_leaves', 'min_samples_leaf'}
def fmt_hp(k, v):   # integer hyperparameters read back from CSV as floats print as integers
    try:
        return str(int(float(v))) if k in INT_HP and float(v).is_integer() else str(v)
    except ValueError:
        return str(v)
def same(row, cfg):
    return all(str(row[k]) == str(v) or (isinstance(v, (int, float)) and not isinstance(v, bool) and str(row[k]) not in ('None', 'sqrt', 'nan')
                                         and abs(float(row[k]) - float(v)) < 1e-12) or (v is None and str(row[k]) in ('None', 'nan'))
               for k, v in cfg.items())
rows, SEARCH_INFO = [], {}
for model, (path, keys, sel, default) in SEARCH.items():
    s = pd.read_csv(path, dtype={k: str for k in keys} if model == 'Random Forest' else None, keep_default_na=(model != 'Random Forest'))
    s = s.sort_values(['val_f1', 'val_ap'], ascending=False).reset_index(drop=True)
    is_sel = s.apply(lambda r: same(r, sel), axis=1); assert is_sel.sum() == 1, model
    is_def = s.apply(lambda r: same(r, default), axis=1) if default else pd.Series(False, index=s.index)
    keep = sorted(set(range(min(8, len(s)))) | set(np.flatnonzero(is_sel)) | set(np.flatnonzero(is_def)))
    SEARCH_INFO[model] = dict(configs=len(s), selected_rank=int(np.flatnonzero(is_sel)[0]) + 1,
                              default_rank=int(np.flatnonzero(is_def)[0]) + 1 if is_def.any() else None)
    rows += ['\\midrule' if rows else None, f"\\multicolumn{{5}}{{l}}{{\\emph{{{model}}} ({len(s)} configurations)}} \\\\"]
    for i in keep:
        r = s.loc[i]
        cfg = ', '.join(f"{ABBR[k]} {fmt_hp(k, r[k])}" for k in keys)
        mark = (' *' if is_sel[i] else '') + (' (default)' if is_def[i] else '')
        rows.append([str(i + 1), cfg + mark, f"{r['val_f1']:.4f}", f"{r['val_ap']:.4f}", cnt(r['rounds']) if 'rounds' in s.columns else '--'])
rows = [r for r in rows if r is not None]
write_table('rev_search_top', 'rlrrr', ['Rank', 'Configuration', 'Val. F1', 'Val. AP', 'Rounds'], rows,
            'Validation-only searches: top 8 configurations by validation F1 (ties by AP), the library default where in the grid, and the selected configuration (*). Full grids: the CSV files.',
            ['02_hyperparameter_search', '13_random_forest_sybil'],
            extra_files=['results/02_search_xgboost.csv', 'results/02_search_lightgbm.csv', 'results/13_search_random_forest.csv'])
print(SEARCH_INFO)

## 11. Named scalars for the running text (`rev_text_numbers.json`)

In [ ]:
NUM, MISSING = {}, []
def put(name, value, src, nb=None):
    if value is None:
        MISSING.append(name); print('MISSING:', name, '-', src); return
    if isinstance(value, (np.generic,)):
        value = value.item()
    NUM[name] = dict(value=value, source=src, platform=platform_of(nb) if nb else 'computed here from data/ (no model)')
r00 = R['00_data_pipeline']
put('n_addresses', r00['n_addresses'], 'results/00 -> n_addresses', '00_data_pipeline')
put('n_sybil', r00['n_sybil'], 'results/00 -> n_sybil', '00_data_pipeline')
put('prevalence', r00['n_sybil'] / r00['n_addresses'], 'results/00 -> n_sybil / n_addresses', '00_data_pipeline')
put('n_negatives', r00['n_addresses'] - r00['n_sybil'], 'results/00 -> n_addresses - n_sybil', '00_data_pipeline')
put('provision_edges_after_cutoff', r00['provision_edges_after_cutoff'], 'results/00 -> provision_edges_after_cutoff', '00_data_pipeline')
for k, nm in [('n', 'n_trees'), ('multi_wallet', 'n_multi_wallet_trees'), ('largest', 'largest_tree'), ('sybils_in_multi', 'sybils_in_multi_wallet_trees')]:
    put(nm, r00['provision_trees'][k], f'results/00 -> provision_trees.{k}', '00_data_pipeline')
for part, x in r00['splits'].items():
    put(f'split_{part.lower()}_rows', x['Total'], f'results/00 -> splits.{part}.Total', '00_data_pipeline')
    put(f'split_{part.lower()}_sybils', x['Sybil'], f'results/00 -> splits.{part}.Sybil', '00_data_pipeline')
for k, v in LARGE_TREES.items():
    put(f'large_trees_over_100_{k}', v, 'computed from sp.build_master_df: gas provision trees with more than 100 wallets')
r07 = R['07_sensitivity_label_vintage']
for k in ['labeled_current', 'labeled_presnapshot', 'wallets_changed']:
    put(k, r07[k], f'results/07 -> {k}', '07_sensitivity_label_vintage')
for m in ['test_f1', 'test_ap', 'test_auroc', 'val_f1']:
    put(f'vintage_{m}_diff_mean', r07['diff_mean'][m], f'results/07 -> diff_mean.{m}', '07_sensitivity_label_vintage')
    put(f'vintage_{m}_diff_sd', r07['diff_sd'][m], f'results/07 -> diff_sd.{m}', '07_sensitivity_label_vintage')
il = r16['initial_list']; c16 = r16['counts']
put('initial_list_rows', il['rows'], 'results/16 -> initial_list.rows', '16_label_robustness_initial_list')
put('initial_list_sha256', il['sha256'], 'results/16 -> initial_list.sha256', '16_label_robustness_initial_list')
put('initial_list_interactors', c16['interactors_on_list'], 'results/16 -> counts.interactors_on_list', '16_label_robustness_initial_list')
put('initial_list_share_of_negatives', c16['share_of_negatives'], 'results/16 -> counts.share_of_negatives', '16_label_robustness_initial_list')
put('initial_list_share_of_interactors', c16['interactors_on_list'] / c16['interactors'], 'results/16 -> counts', '16_label_robustness_initial_list')
fpv = [x for x in r16['fp_composition'] if x['note']][0]
for k in ['threshold', 'fp', 'fp_on_initial_list', 'fp_not_on_list', 'fp_share_on_list', 'precision', 'precision_if_list_counted_sybil']:
    put(f'initial_list_valthr_{k}', fpv[k], f'results/16 -> fp_composition[validation threshold].{k}', '16_label_robustness_initial_list')
for x in r16['table_2x2']:
    tag = ('baseline' if x['model'] == 'Baseline' else 'clean') + '_' + ('full' if x['test_set'] == 'full test' else 'nolist')
    for m in ['f1', 'ap', 'auroc', 'precision', 'recall']:
        put(f'initial_list_2x2_{tag}_{m}', x[m], f"results/16 -> table_2x2[{x['model']}, {x['test_set']}].{m}", '16_label_robustness_initial_list')
for k in ['delta_clean_f1_mean', 'delta_clean_f1_std', 'delta_clean_f1_higher_n', 'delta_full_f1_mean', 'delta_full_f1_std']:
    put(f'initial_list_10split_{k}', r16['summary'].get(k), f'results/16 -> summary.{k}', '16_label_robustness_initial_list')
for k, v in BOUNTY.items():
    put(f'bounty_{k}', v, f'computed from {SYB_PATH} matched to the interactors')
put('level1_tree_overlap', int(L1['tree']), 'results/15 -> report_overlap.per_split[random, 42].tree', '15_split_tree_report')
put('level1_report_overlap', int(L1['report']), 'results/15 -> report_overlap.per_split[random, 42].report', '15_split_tree_report')
put('level2_tree_overlap', int(L2['tree']), 'results/15 -> report_overlap.per_split[tree, 42].tree', '15_split_tree_report')
put('level2_report_overlap', int(L2['report']), 'results/15 -> report_overlap.per_split[tree, 42].report', '15_split_tree_report')
put('level2_report_overlap_10seed_mean', ii['report_overlap_tree_10seeds_mean'], 'results/15 -> interpretation_inputs', '15_split_tree_report')
put('level2_report_overlap_10seed_sd', ii['report_overlap_tree_10seeds_sd'], 'results/15 -> interpretation_inputs', '15_split_tree_report')
put('test_sybils_seed42', ii['test_sybils'], 'results/15 -> interpretation_inputs.test_sybils', '15_split_tree_report')
put('large_components_rows', r15['composition']['big_rows'], 'results/15 -> composition.big_rows', '15_split_tree_report')
put('large_components_sybils', r15['composition']['big_sybils'], 'results/15 -> composition.big_sybils', '15_split_tree_report')
put('large_components_share_of_sybils', r15['composition']['big_sybils'] / r00['n_sybil'], 'results/15 -> composition.big_sybils / n_sybil', '15_split_tree_report')
put('level2_train_sybils_from_large_components_share', cs22['tree_split_big_share'], 'results/22 -> composition_summary.tree_split_big_share', '22_report_dependence')
put('level3_train_sybils_from_large_components_share', cs22['big_share_held_mean'], 'results/22 -> composition_summary.big_share_held_mean', '22_report_dependence')
put('level3_report_mate_coverage_mean', cs22['mate_coverage_mean'], 'results/22 -> composition_summary.mate_coverage_mean', '22_report_dependence')
put('level3_test_prevalence', cs22['k_a_prevalence_mean'], 'results/22 -> composition_summary.k_a_prevalence_mean', '22_report_dependence')
put('level3_fold_repeats', len({(x['repeat'], x['k']) for x in r22['per_fold']}), 'results/22 -> per_fold (distinct repeat, fold)', '22_report_dependence')
for model in ['LightGBM', 'XGBoost']:
    for m in ['f1', 'ap', 'auroc']:
        t = [x for x in r22['tests'] if x['model'] == model and x['metric'] == m][0]
        for k in ['diff_mean', 'diff_sd', 'seen_higher_n', 'p_corrected']:
            put(f'level3_{model.lower()}_{m}_{k}', t[k], f'results/22 -> tests[{model}, {m}].{k}', '22_report_dependence')
r02 = R['02_hyperparameter_search']
put('xgb_selected', r02['xgb_selected'], 'results/02 -> xgb_selected', '02_hyperparameter_search')
put('lgbm_selected', r02['lgbm_selected'], 'results/02 -> lgbm_selected', '02_hyperparameter_search')
put('search_n_configs', r02['n_configs'], 'results/02 -> n_configs', '02_hyperparameter_search')
put('xgb_rounds_seed42_3seeds', R['03_xgboost_sybil']['rounds'], 'results/03 -> rounds', '03_xgboost_sybil')
put('lgbm_rounds_seed42_3seeds', R['04_lightgbm_sybil']['rounds'], 'results/04 -> rounds', '04_lightgbm_sybil')
r13 = R['13_random_forest_sybil']
put('rf_selected', {k: r13['params'][k] for k in ['max_features', 'min_samples_leaf', 'max_depth', 'n_estimators']}, 'results/13 -> params', '13_random_forest_sybil')
put('rf_n_configs', r13['n_configs'], 'results/13 -> n_configs', '13_random_forest_sybil')
wts = [x['xgb_weight'] for x in r17['split_info']]
put('blend_weight_min', min(wts), 'results/17 -> split_info.xgb_weight', '17_models_10_splits')
put('blend_weight_max', max(wts), 'results/17 -> split_info.xgb_weight', '17_models_10_splits')
put('blend_weight_mean', float(np.mean(wts)), 'results/17 -> split_info.xgb_weight', '17_models_10_splits')
for x in r17['cost']['table']:
    key = x['model'].lower().replace(' ', '_')
    for k in ['train_seconds_single_median', 'train_seconds_3seed', 'score_seconds_test_median', 'score_rows_per_second', 'pickled_mib_median', 'peak_rss_increase_mib_median']:
        put(f'cost_{key}_{k}', x[k], f"results/17 -> cost.table[{x['model']}].{k}", '17_models_10_splits')
put('cost_build_master_df_seconds', r17['cost']['build_master_df_seconds'], 'results/17 -> cost.build_master_df_seconds', '17_models_10_splits')
for model in ['LightGBM', 'XGBoost', 'Random Forest', 'Logistic regression', 'Ensemble']:
    for m in ['f1', 'ap', 'auroc', 'precision', 'recall']:
        key = model.lower().replace(' ', '_')
        put(f'models10_{key}_{m}_mean', r17['summary'][model][f'{m}_mean'], f'results/17 -> summary.{model}.{m}_mean', '17_models_10_splits')
        put(f'models10_{key}_{m}_sd', r17['summary'][model][f'{m}_std'], f'results/17 -> summary.{model}.{m}_std', '17_models_10_splits')
TEXT_NUMBERS = dict(generated_by=f'25_paper_tables.ipynb at {sp.CODE_COMMIT}', numbers=NUM, missing=MISSING,
                    per_category_note=PERCAT_NOTE)
with open(os.path.join(TAB_DIR, 'rev_text_numbers.json'), 'w') as f:
    json.dump(TEXT_NUMBERS, f, indent=2, default=lambda o: o.item() if isinstance(o, np.generic) else str(o))
TABLES['rev_text_numbers'] = dict(file=f'{TAB_DIR}/rev_text_numbers.json', sources='per name, inside the file', platform_note=None)
print(f'{len(NUM)} named numbers; missing: {MISSING}')

## Checks

In [ ]:
# levels table equals interpretation_inputs (asserted above); per-category sums asserted above; taxonomy totals asserted above
# stored p-values of 19 appear unchanged in their table
lines6 = t6.splitlines()
for i, w in enumerate(PROV):
    idx = [j for j, l in enumerate(lines6) if l.startswith(w + ' &')][0]
    got = cells_of(lines6[idx + 1])[1:]
    want = [f"{int(PROV[w][c]['all_higher_n'])}/10; {pval(PROV[w][c]['p_corrected'])}" for c in CATS]
    assert got == want, (w, got, want)
# every .tex table: column structure; compile if pdflatex exists
for name, t in TABLES.items():
    if not t['file'].endswith('.tex'):
        continue
    text = open(t['file']).read()
    ncol = len(re.search(r'\\begin\{tabular\}\{([a-z]+)\}', text).group(1))
    for l in text.splitlines():
        if l.endswith('\\\\'):
            spans = sum(int(s) - 1 for s in re.findall(r'\\multicolumn\{(\d+)\}', l))
            assert len(re.findall(r'(?<!\\)&', l)) + 1 + spans == ncol, (name, l)
if shutil.which('pdflatex'):
    with tempfile.TemporaryDirectory() as d:
        body = ''.join(f'\\input{{{os.path.abspath(t["file"])}}}\n\\bigskip\n' for t in TABLES.values() if t['file'].endswith('.tex'))
        open(os.path.join(d, 'w.tex'), 'w').write('\\documentclass{article}\n\\usepackage{booktabs}\n\\begin{document}\n' + body + '\\end{document}\n')
        assert subprocess.run(['pdflatex', '-interaction=nonstopmode', 'w.tex'], cwd=d, capture_output=True).returncode == 0
    COMPILE = 'pdflatex: all tables compile'
else:
    COMPILE = 'WARNING: pdflatex not installed; compile check skipped (structural column check passed)'
print(COMPILE)
assert not MISSING, MISSING
print('Tables:', sorted(TABLES))

## Save the results record

In [ ]:
sp.save_results([], '25_paper_tables', extra=dict(
    tables=TABLES, sources={k: v['sources'] for k, v in TABLES.items()},
    platforms={n: platform_of(n) for n in SOURCES}, compile_check=COMPILE,
    bounty_stats=dict(computed=BOUNTY, expected=EXPECTED), large_trees=LARGE_TREES, search=SEARCH_INFO,
    fields_used_temporal=FIELDS20, per_category_note=PERCAT_NOTE, results14_flag=FLAG14, results14_platform=plat14,
    text_numbers_missing=MISSING, platform=PLATFORM))